# Baumdaten für's Klimadashboard

Aktuell: Sehr grobe Annäherung der Werte durch pauschale Faktoren!

**Empfehlung für ein kommunales Klimadashboard:** \
Wenn das Dashboard öffentlich genutzt wird, ist es sinnvoll, die Berechnungen auf etablierte Methoden zu stützen und die Kennzahlen transparent als modellierte Schätzwerte auszuweisen. Mit euren vorhandenen Daten (artbotanisch, Höhe und Kronendurchmesser) lässt sich zudem ein deutlich genaueres Modell aufbauen, das artspezifische Wachstumsunterschiede berücksichtigt und näher an Verfahren wie denen aus i-Tree oder städtischen Baumkatastern liegt. Das wäre für die CO₂-Bindung, Sauerstoffproduktion und Kühlwirkung fachlich robuster als pauschale Faktoren.

In [ ]:
# Fehlende Pakete installieren, ggf. einkommentieren und ausführen

#!python -m pip install geopandas
#!python -m pip install pandas
#!python -m pip install matplotlib
#!python -m pip install dotenv

In [ ]:
import requests
import xml.etree.ElementTree as ET

import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt
import os
from dotenv import load_dotenv
from pprint import pprint
import json
import os 

# Place .env file in the same directory as this script and add the following lines:
# WFS_URL=https://your-wfs-url
# USER=api-username
# PASSWORD=api-password
# PROXY=http://your-proxy-url:port (optional)

load_dotenv()
WFS_URL = os.getenv("WFS_URL")
USER = os.getenv("USER")
PASSWORD = os.getenv("PASSWORD")
PROXY = os.getenv("PROXY")
proxies = {"http": PROXY, "https": PROXY} if PROXY else False

if not WFS_URL:
    print("Fehler: WFS_URL ist nicht gesetzt.")
if not USER:
    print("Fehler: USER ist nicht gesetzt.")
if not PASSWORD:
    print("Fehler: PASSWORD ist nicht gesetzt.")




CURRENT_YEAR = pd.Timestamp.now().year
GEOJSON_FILE = "data/feature_data.geojson"
layer_name = "tree:stamuenster_point"

params = {
    "service": "WFS",
    "version": "2.0.0",
    "request": "GetFeature",
    "typeNames": layer_name,
    "outputFormat": "application/json",
}

if os.path.exists(GEOJSON_FILE):
    print(f"Datei '{GEOJSON_FILE}' existiert bereits.")
else:
    print(f"Datei wird gespeichert als: {GEOJSON_FILE}")
 
    session = requests.Session()
    session.auth = (USER, PASSWORD)
    response = session.get(WFS_URL, params=params, proxies=proxies)

    with open(GEOJSON_FILE, "w") as f:
        json.dump(response.json(), f)

# echo filesize of geojson file in megabytes
print(f"Größe der GeoJSON-Datei: {os.path.getsize(GEOJSON_FILE) / (1024 * 1024):.2f} MB")
gdf_orig = gpd.read_file(GEOJSON_FILE)
print("Spalten: ", gdf_orig.columns)


# Ausreisser entfernen

In [ ]:

CHECK_FIELD = "kronendurchmesser"

print("Kronendurchmesser über 100 Meter herausfiltern")
ausreisser = gdf_orig[gdf_orig["kronendurchmesser"] > 100]
ausreisser = ausreisser.sort_values(by=CHECK_FIELD, ascending=False)
print(ausreisser[["artdeutsch", CHECK_FIELD]])

gdf = gdf_orig.drop(ausreisser.index)

# Beispiel: Alter der Bäume in Klassen aufteilen

In [ ]:
def add_labels(plt, x, y):
    for i in range(len(x)):
        plt.text(i, y[i] // 2, y[i], ha='center')  # Placing text slightly above the bar
        
current_year = pd.Timestamp.now().year

birth_year = pd.to_numeric(gdf["pflanzjahr"], errors="coerce")
birth_year = birth_year[birth_year.between(1800, current_year)]

ages = current_year - birth_year
max_age = int(ages.max())
if max_age <= 100:
    bins = [0, 20, 40, 60, 80, 100]
else:
    bins = [0, 20, 40, 60, 80, 100, max_age + 1]
age_categories = pd.cut(ages, bins=bins, right=False, include_lowest=True)
age_counts = age_categories.value_counts().sort_index()

fig, ax = plt.subplots(figsize=(5, 5))
ax.set_xlabel("Alter in Jahren")
ax.set_ylabel("Anzahl Bäume")

ax.set_title("Alter der Bäume (20-Jahres-Kategorien) aus Pflanzjahr")


age_counts.plot(kind="barh", ax=ax, color="steelblue")

# Berechnung der Werte für's Klimadashboard

In [ ]:
import numpy as np
import pandas as pd


# Altersgruppendefinition für's Klimadashboard
def altersgruppe(alter):
    if alter < 15:
        return "Jung"
    elif alter < 80:
        return "Mittel"
    else:
        return "Alt"
    

# Formel für: CO2 Bindung
def co2_rate(alter):
    if alter < 15:
        return 10   # kg/Jahr
    elif alter < 40:
        return 25
    else:
        return 40

gdf["alter"] = CURRENT_YEAR - gdf["pflanzjahr"]


# =========================== Berechnungen ===========================

# Kronenschirmfläche = Kreisfläche = π * r²
gdf["kronenschirmflaeche"] = np.pi * (gdf["kronendurchmesser"] / 2) ** 2

# CO2 Bindung in kg pro Jahr (Formel siehe Funktion co2_rate)
gdf["co2_kg_pro_jahr"] = gdf["alter"].apply(co2_rate)
gdf["o2_kg_pro_jahr"] = gdf["co2_kg_pro_jahr"] * (32 / 12)

# Kühlleistung: 0.15 kW  pro m² Kronenschirmfläche
gdf["kuehlleistung_kw"] = gdf["kronenschirmflaeche"] * 0.15

# TODO - Pflanzungen pro Jahr => Alle Bäume mit Alter 0 ?
# TODO - Anzahl der Fällungen => Bei 67 nachfragen 

def berechne_jahr(df, jahr):
    temp = df.copy()

    # Alter im betrachteten Jahr
    temp["alter"] = jahr - temp["pflanzjahr"]

    # Bäume, die damals noch nicht gepflanzt waren, ausblenden
    # ACHTUNG - TODO - Was ist mit gefällten Bäumen?
    temp = temp[temp["alter"] >= 0]

    # Altersgruppen erstellen
    temp["gruppe"] = temp["alter"].apply(altersgruppe)

    # Hilfsfunktion, um die %-Werte für ein bestimmtes Feld zu berechnen und dem DataFrame hinzuzufügen
    def addFieldsToDataFrame(df, FIELD):
        erg = (
            temp.groupby("gruppe")[FIELD]
            .sum()
            .reindex(["Jung", "Mittel", "Alt"], fill_value=0)
        )
    
        gesamt = erg.sum()
        vals= erg.values
        percent = erg.values / gesamt * 100
        df["gruppe"] = erg.index
        df[f"{FIELD}"] = vals 
        df[f"{FIELD}_prozent"] = percent
        return df


    # Alle Werte des aktuellen Jahres zum DataFrame hinzufügen
    temp_df = pd.DataFrame()
    temp_df["jahr"] = jahr
    temp_df = addFieldsToDataFrame(temp_df, "kronenschirmflaeche")
    temp_df = addFieldsToDataFrame(temp_df, "co2_kg_pro_jahr")
    temp_df = addFieldsToDataFrame(temp_df, "o2_kg_pro_jahr")
    temp_df = addFieldsToDataFrame(temp_df, "kuehlleistung_kw")
    return temp_df

startjahr = 2020
endjahr = CURRENT_YEAR

alle = []

for jahr in range(startjahr, endjahr + 1):
    t = berechne_jahr(gdf, jahr)
    t["jahr"] = jahr
    alle.append(t)

# Dataframe erstellen
slider_df = pd.concat(alle, ignore_index=True)

# Runden der Werte auf 2 Nachkommastellen, außer für die Spalten "jahr" und "gruppe"
cols = {col: 2 for col in slider_df.columns if col not in ("jahr", "gruppe")}
slider_df = slider_df.round(cols)

# Daten als CSV speichern
slider_df.to_csv("data/baumkachel_daten.csv", index=False)

# Daten ausgeben
slider_df


# Plotten der Werte 

In [ ]:

def plot_stacked_bar(df, MYVAL):
    pivot_plot = df.pivot(index="jahr", columns="gruppe", values=MYVAL)

    colors = {"Jung": "#4CAF50", "Mittel": "#2196F3", "Alt": "#FF9800"}

    ax = pivot_plot.plot(
        kind="bar",
        stacked=True,
        color=[colors.get(col, "#888888") for col in pivot_plot.columns],
        figsize=(8, 5),
    )

    ax.set_xlabel("Jahr")
    ax.set_ylabel("Anteil in %")
    ax.set_title(f"Gestapelter Anteil der Altersgruppen an der {MYVAL} pro Jahr")
    ax.set_ylim(0, 100)
    ax.legend(title="Altersgruppe")
    plt.tight_layout()


plot_stacked_bar(slider_df, "kronenschirmflaeche_prozent")
plot_stacked_bar(slider_df, "co2_kg_pro_jahr_prozent")
plot_stacked_bar(slider_df, "o2_kg_pro_jahr_prozent")
plot_stacked_bar(slider_df, "kuehlleistung_kw_prozent")


